# 03 - Local Metadata Assignment & RBAC Validation

**Phase 1, Step 3** of the Ingestion & Chunking Lifecycle.

**Objective:** For each chunk produced by the 4 strategies, assign local metadata (`contains_PII`, `page_number`, `chunk_id`) and validate that the RBAC security model holds:

- A chunk with PII should **never** have `clearance_level` = 0 (public)
- Chunks with credentials/IBAN must be `clearance_level` >= 3 (strict)
- Department filtering must be set for `clearance_level` >= 2

This notebook provides the **measurable results** needed before moving to Phase 2.

In [1]:
import json
import re
from collections import defaultdict

import pandas as pd
from langchain.schema import Document

# Load chunk results from notebook 02
with open("../../../data/results/notebook_results/chunk_results.json", "r", encoding="utf-8") as f:
    raw_chunks = json.load(f)

all_chunk_results: dict[str, dict[str, list[Document]]] = {}
for strategy, docs_dict in raw_chunks.items():
    all_chunk_results[strategy] = {}
    for filename, chunk_list in docs_dict.items():
        all_chunk_results[strategy][filename] = [
            Document(page_content=c["page_content"], metadata=c["metadata"])
            for c in chunk_list
        ]

# Load metrics from notebook 02
df_metrics = pd.read_csv("../../../data/results/notebook_results/chunking_metrics.csv")

strategies = list(all_chunk_results.keys())
print(f"Loaded strategies: {strategies}")
for s in strategies:
    total = sum(len(v) for v in all_chunk_results[s].values())
    print(f"  {s}: {total} chunks")

Loaded strategies: ['fixed', 'structural', 'semantic', 'custom_rbac']
  fixed: 20 chunks
  structural: 18 chunks
  semantic: 74 chunks
  custom_rbac: 33 chunks


## 1. PII Detection Engine

Same patterns from notebook 02, applied systematically to assign `contains_PII` and `sensitivity_types` to every chunk.

In [2]:
PII_PATTERNS = {
    "email": r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
    "iban": r"[A-Z]{2}\d{2}[\s]?[A-Z0-9]{4}[\s]?[\d]{4}[\s]?[\d]{4}[\s]?[\d]{4}[\s]?[\d]{4}[\s]?[\d]{0,3}",
    "ip_address": r"\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b",
    "password": r"(?i)(?:password|contrase[ñn]a|pwd|override[_ ]?password)\s*[:=]\s*['\"]?([^\s'\"]+)",
    "phone": r"\+?\d{1,3}[\s-]?\d{3,4}[\s-]?\d{3,4}[\s-]?\d{0,4}",
    "person_name_context": r"(?:D\.|Mr\.|Mrs\.|Ms\.|Dr\.|Contacto:?)\s+[A-Z][a-záéíóúñ]+\s+[A-Z][a-záéíóúñ]+",
    "client_id": r"CLI-\d{3,}",
    "swift_code": r"SWIFT[:\s]*[A-Z]{4}[A-Z]{2}[A-Z0-9]{2,5}",
    "monetary_value": r"€\s?[\d.,]+(?:\s?(?:million|mil))?",
}

# Minimum clearance required when PII type is detected
PII_CLEARANCE_RULES = {
    "password": 3,
    "iban": 3,
    "swift_code": 3,
    "email": 2,
    "ip_address": 2,
    "client_id": 2,
    "person_name_context": 2,
    "phone": 1,
    "monetary_value": 1,
}


def detect_pii(text: str) -> dict[str, list[str]]:
    findings = {}
    for name, pattern in PII_PATTERNS.items():
        matches = re.findall(pattern, text)
        if matches:
            findings[name] = matches
    return findings


def required_clearance_for_pii(pii_types: list[str]) -> int:
    """Return the minimum clearance level required given the PII types found."""
    if not pii_types:
        return 0
    return max(PII_CLEARANCE_RULES.get(t, 0) for t in pii_types)


print("PII detection engine ready.")
print(f"Clearance rules: {PII_CLEARANCE_RULES}")

PII detection engine ready.
Clearance rules: {'password': 3, 'iban': 3, 'swift_code': 3, 'email': 2, 'ip_address': 2, 'client_id': 2, 'person_name_context': 2, 'phone': 1, 'monetary_value': 1}


## 2. Assign Local Metadata to All Chunks

For strategies that don't already have local metadata (fixed, structural, semantic), we assign it now.

In [3]:
def enrich_local_metadata(chunks: list[Document]) -> list[Document]:
    """Add local metadata (contains_PII, sensitivity_types) to chunks that don't have it."""
    for chunk in chunks:
        # Skip if already enriched (custom_rbac strategy)
        if "contains_PII" in chunk.metadata:
            continue
        
        pii = detect_pii(chunk.page_content)
        chunk.metadata["contains_PII"] = bool(pii)
        chunk.metadata["sensitivity_types"] = sorted(pii.keys()) if pii else []
    return chunks


# Enrich all strategies
for strategy, docs_dict in all_chunk_results.items():
    for filename, chunks in docs_dict.items():
        enrich_local_metadata(chunks)

print("Local metadata assigned to all chunks across all strategies.")

Local metadata assigned to all chunks across all strategies.


## 3. RBAC Security Validation

Verify that the security model holds for each strategy. A **violation** occurs when:
- A chunk contains PII but its `clearance_level` is lower than what the PII type requires
- A chunk has `clearance_level >= 2` but `allowed_departments` is not set

In [4]:
def validate_rbac(chunks: list[Document], strategy: str, filename: str) -> list[dict]:
    """Check RBAC constraints. Returns list of violations."""
    violations = []
    
    for chunk in chunks:
        pii = detect_pii(chunk.page_content)
        current_clearance = chunk.metadata.get("clearance_level", 0)
        required_clearance = required_clearance_for_pii(list(pii.keys()))
        
        # Violation 1: Clearance too low for PII content
        if pii and current_clearance < required_clearance:
            violations.append({
                "strategy": strategy,
                "document": filename,
                "chunk_id": chunk.metadata.get("chunk_id", "?"),
                "violation": "clearance_too_low",
                "detail": f"Has {list(pii.keys())} but clearance={current_clearance}, needs>={required_clearance}",
                "severity": "CRITICAL" if required_clearance == 3 else "WARNING",
            })
        
        # Violation 2: Missing department for restricted content
        if current_clearance >= 2:
            dept = chunk.metadata.get("allowed_departments", "all")
            if dept == "all" or dept is None:
                violations.append({
                    "strategy": strategy,
                    "document": filename,
                    "chunk_id": chunk.metadata.get("chunk_id", "?"),
                    "violation": "missing_department",
                    "detail": f"clearance={current_clearance} but allowed_departments='{dept}'",
                    "severity": "WARNING",
                })
    
    return violations


# Run validation across all strategies and documents
all_violations: list[dict] = []

for strategy, docs_dict in all_chunk_results.items():
    for filename, chunks in docs_dict.items():
        violations = validate_rbac(chunks, strategy, filename)
        all_violations.extend(violations)

df_violations = pd.DataFrame(all_violations)

if df_violations.empty:
    print("No RBAC violations found across any strategy.")
else:
    print(f"Total violations found: {len(df_violations)}")
    print(f"\nCRITICAL violations: {len(df_violations[df_violations['severity'] == 'CRITICAL'])}")
    print(f"WARNING violations: {len(df_violations[df_violations['severity'] == 'WARNING'])}")

Total violations found: 13

CRITICAL violations: 6
WARNING violations: 7


In [5]:
# Detailed violation report
if not df_violations.empty:
    print("=" * 90)
    print("RBAC VIOLATION REPORT")
    print("=" * 90)
    
    for strategy in strategies:
        strategy_v = df_violations[df_violations["strategy"] == strategy]
        print(f"\n--- {strategy.upper()} ---")
        if strategy_v.empty:
            print("  No violations.")
        else:
            print(f"  Violations: {len(strategy_v)}")
            # Group by violation type
            for vtype in strategy_v["violation"].unique():
                subset = strategy_v[strategy_v["violation"] == vtype]
                print(f"    [{vtype}] x{len(subset)}")
                for _, row in subset.head(3).iterrows():
                    print(f"      {row['document']} / {row['chunk_id']}: {row['detail']}")
                if len(subset) > 3:
                    print(f"      ... and {len(subset) - 3} more")

RBAC VIOLATION REPORT

--- FIXED ---
  Violations: 4
    [clearance_too_low] x4
      Witty-QuickGuide-EN.pdf / fixed_000: Has ['phone'] but clearance=0, needs>=1
      Witty-QuickGuide-EN.pdf / fixed_001: Has ['email', 'phone'] but clearance=0, needs>=2
      distribution-contract-2026.docx / fixed_002: Has ['iban', 'phone', 'swift_code'] but clearance=2, needs>=3
      ... and 1 more

--- STRUCTURAL ---
  Violations: 3
    [clearance_too_low] x3
      Witty-QuickGuide-EN.pdf / struct_000: Has ['email', 'phone'] but clearance=0, needs>=2
      distribution-contract-2026.docx / struct_003: Has ['iban', 'phone', 'swift_code'] but clearance=2, needs>=3
      server_logs_witty_backend.txt / struct_004: Has ['password'] but clearance=2, needs>=3

--- SEMANTIC ---
  Violations: 4
    [clearance_too_low] x4
      Witty-QuickGuide-EN.pdf / semantic_003: Has ['phone'] but clearance=0, needs>=1
      Witty-QuickGuide-EN.pdf / semantic_004: Has ['email', 'phone'] but clearance=0, needs>=2
      

## 4. Violation Summary by Strategy

This is the key result: which strategy best protects sensitive data?

In [6]:
print("=" * 90)
print("SECURITY SCORECARD BY STRATEGY")
print("=" * 90)

scorecard_rows = []

for strategy in strategies:
    total_chunks = sum(len(v) for v in all_chunk_results[strategy].values())
    chunks_with_pii = 0
    chunks_pii_isolated = 0  # PII in small, dedicated chunk
    
    for filename, chunks in all_chunk_results[strategy].items():
        for chunk in chunks:
            pii = detect_pii(chunk.page_content)
            if pii:
                chunks_with_pii += 1
                # Consider isolated if marked contains_PII or chunk is small
                if chunk.metadata.get("contains_PII") and len(chunk.page_content) < 300:
                    chunks_pii_isolated += 1
    
    strategy_violations = df_violations[df_violations["strategy"] == strategy] if not df_violations.empty else pd.DataFrame()
    critical = len(strategy_violations[strategy_violations["severity"] == "CRITICAL"]) if not strategy_violations.empty else 0
    warnings = len(strategy_violations[strategy_violations["severity"] == "WARNING"]) if not strategy_violations.empty else 0
    
    isolation_ratio = chunks_pii_isolated / chunks_with_pii if chunks_with_pii > 0 else 1.0
    
    scorecard_rows.append({
        "Strategy": strategy,
        "Total Chunks": total_chunks,
        "PII Chunks": chunks_with_pii,
        "PII Isolated": chunks_pii_isolated,
        "Isolation %": f"{isolation_ratio:.0%}",
        "CRITICAL Violations": critical,
        "WARNING Violations": warnings,
        "Security Score": "PASS" if critical == 0 else "FAIL",
    })

df_scorecard = pd.DataFrame(scorecard_rows)
print(df_scorecard.to_string(index=False))

SECURITY SCORECARD BY STRATEGY
   Strategy  Total Chunks  PII Chunks  PII Isolated Isolation %  CRITICAL Violations  WARNING Violations Security Score
      fixed            20          10             3         30%                    2                   2           FAIL
 structural            18           9             3         33%                    2                   1           FAIL
   semantic            74          11             9         82%                    2                   2           FAIL
custom_rbac            33          19            15         79%                    0                   2           PASS


## 5. Deep Dive: Custom RBAC Strategy Inspection

Detailed view of how the custom strategy handles each document's sensitive content.

In [7]:
print("=" * 90)
print("CUSTOM RBAC STRATEGY - DETAILED CHUNK INSPECTION")
print("=" * 90)

if "custom_rbac" in all_chunk_results:
    for filename, chunks in all_chunk_results["custom_rbac"].items():
        print(f"\n{'─' * 70}")
        print(f"FILE: {filename}")
        print(f"Total chunks: {len(chunks)}")
        
        for i, chunk in enumerate(chunks):
            pii = detect_pii(chunk.page_content)
            pii_flag = f" ** PII: {list(pii.keys())} **" if pii else ""
            cl = chunk.metadata.get('clearance_level', '?')
            dept = chunk.metadata.get('allowed_departments', 'all')
            is_pii = chunk.metadata.get('contains_PII', False)
            
            marker = "[SENSITIVE]" if is_pii else "[CLEAN]    "
            print(f"\n  {marker} Chunk {i} | {len(chunk.page_content)} chars | clearance={cl} | dept={dept}{pii_flag}")
            preview = chunk.page_content[:150].replace('\n', ' ')
            print(f"    > {preview}")

CUSTOM RBAC STRATEGY - DETAILED CHUNK INSPECTION

──────────────────────────────────────────────────────────────────────
FILE: Witty-QuickGuide-EN.pdf
Total chunks: 3

  [SENSITIVE] Chunk 0 | 17 chars | clearance=2 | dept=all ** PII: ['email'] **
    > info@microgate.it

  [CLEAN]     Chunk 1 | 485 chars | clearance=0 | dept=all ** PII: ['phone'] **
    > ENG QUICK GUIDE Witty Manager Software The USB stick contains the Witty Manager software, as well as the relevant user manual, which you should open o

  [CLEAN]     Chunk 2 | 1743 chars | clearance=0 | dept=all
    > Documentation The Witty Kit user manual is stored on the USB stick  inside the backpack pocket. Please open or print the  PDF file referred to in this

──────────────────────────────────────────────────────────────────────
FILE: Witty-Financial-Report-2025.pdf
Total chunks: 5

  [CLEAN]     Chunk 0 | 215 chars | clearance=3 | dept=finance
    > MICROGATE S.R.L. - QUARTERLY FINANCIAL REPORT (Q3 2025)  Department: Finance 

## 6. Export Final Results

Export the complete analysis for Phase 2 evaluation.

In [8]:
# Export violation report
if not df_violations.empty:
    df_violations.to_csv("../../data/results/notebook_results/rbac_violations.csv", index=False)
    print("Violations exported to ../../data/results/notebook_results/rbac_violations.csv")
else:
    print("No violations to export.")

# Export scorecard
df_scorecard.to_csv("../../data/results/notebook_results/security_scorecard.csv", index=False)
print("Security scorecard exported to ../../data/results/notebook_results/security_scorecard.csv")

# Summary
print("\n" + "=" * 90)
print("PHASE 1 COMPLETE - Summary")
print("=" * 90)
print(f"Documents processed: {len(all_chunk_results[strategies[0]])}")
print(f"Strategies evaluated: {strategies}")
print(f"Total RBAC violations: {len(all_violations)}")
print("\nReady for Phase 2: Qualitative and Quantitative Analysis.")

Violations exported to ../data/rbac_violations.csv
Security scorecard exported to ../data/security_scorecard.csv

PHASE 1 COMPLETE - Summary
Documents processed: 5
Strategies evaluated: ['fixed', 'structural', 'semantic', 'custom_rbac']
Total RBAC violations: 13

Ready for Phase 2: Qualitative and Quantitative Analysis.
